In [1]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
try:
    user_secrets = UserSecretsClient()
    secret_value_0 = user_secrets.get_secret("HF_TOKEN")

    login(secret_value_0)
    print("Logged in to Hugging Face successfully!")
except Exception as e:
    print(f"Error logging in: {e}")
    print("Make sure you added the secret correctly in Add-ons -> Secrets")

Logged in to Hugging Face successfully!


In [2]:
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from tqdm import tqdm


MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

print("Loading Model in Float16 (Standard Precision)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,  #
    device_map="auto",
    trust_remote_code=True
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=600, 
    do_sample=True,
    temperature=0.8,
    top_p=0.95,
    repetition_penalty=1.15
)

try:
    df = pd.read_csv("/kaggle/input/hc3-project-cleaned-essays/filtered_essay_corpus.csv")
    source_essays = df[df['generated'] == 0].sample(500, random_state=42)
except:
    print("Dataset not found. Creating dummy topics.")
    source_essays = pd.DataFrame({'text': ["The history of the Roman Empire is complex...", "Climate change is a pressing issue..."]})

adversarial_data = []

def get_few_shot_examples(df, n_shots=2):
    examples = df.sample(n_shots)['text'].tolist()
    truncated_examples = [" ".join(ex.split()[:150]) + "..." for ex in examples]
    return truncated_examples

print(f"Generating {len(source_essays)} adversarial essays with Few-Shot Prompting...")

for index, row in tqdm(source_essays.iterrows(), total=len(source_essays)):
    topic_snippet = " ".join(str(row['text']).split()[:15])
    
    shots = get_few_shot_examples(source_essays, n_shots=2)
    
    prompt = [
        {
            "role": "system",
            "content": "You are a college student. Analyze the style of the following examples. They use varied sentence lengths, natural transitions, and imperfect grammar. Mimic this style exactly."
        },
        {
            "role": "user",
            "content": f"Example 1: {shots[0]}"
        },
        {
            "role": "user",
            "content": f"Example 2: {shots[1]}"
        },
        {
            "role": "user",
            "content": f"Now, write a 400-word essay starting with: '{topic_snippet}...'"
        }
    ]
    prompt_formatted = tokenizer.apply_chat_template(prompt, tokenize=False, add_generation_prompt=True)
    
    try:
        outputs = pipe(prompt_formatted)
        generated_text = outputs[0]["generated_text"].split("<|im_start|>assistant\n")[-1].strip()
        
        adversarial_data.append({
            "text": generated_text,
            "generated": 1,             
            "source": "adversarial_few_shot",
            "original_topic_source": index
        })
    except Exception as e:
        print(f"Error: {e}")
        continue        

df_adv = pd.DataFrame(adversarial_data)
df_adv.to_csv("adversarial_scratch_set.csv", index=False)
print("Saved 'adversarial_scratch_set.csv'. Use this for the final robustness evaluation.")

2025-12-06 11:42:57.421873: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765021377.606492      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765021377.657178      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Device set to use cuda:0


Generating 500 adversarial essays with Few-Shot Prompting...


100%|██████████| 500/500 [3:03:29<00:00, 22.02s/it]

Saved 'adversarial_scratch_set.csv'. Use this for the final robustness evaluation.
